# NB15 diagnostics — is the T3 gain listening or memorising? (CPU, ~10 min)

Runs on the saved NB15 candidate tables; trains nothing on new data and never touches the test set.
These are **diagnostics**: reported, never used to tune the final system.

1. Sentence overlap between Extra-dev and the training folds.
2. Label audit: every Extra-dev candidate's label vs what the official counting implies; 20 printed examples (spec risk response, AUC < 0.75).
3. Feature-group ablation: full / no identity features / acoustic only / pair only, same training data and procedure.
4. Deletion rule: revert every deletion flag, keep every substitution; and the verifier with all deletions reverted.

**Run settings:** Accelerator *None*, Internet *On*. Add data: the NB11 output, the NB14 output, the NB15 output.
**After the run:** paste back everything printed by the last cell (**PASTE BACK**).


In [ ]:
# ---- Shared install cell (identical in nb11-nb16) --------------------------------
# pip install git+... does not fetch submodules, and the official scorer is a pinned
# submodule, so clone recursively and install from the clone (SPEC decision 21).
import os, subprocess, sys
REPO_URL = "https://github.com/markeet04/Quran_phenome.git"
REPO_DIR = "/tmp/Quran_phenome"          # outside /kaggle/working: not saved as output
BRANCH = "main"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # only if the repo is private
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = r.stderr if not token else r.stderr.replace(token, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git("clone", "--recursive", "-q", "-b", BRANCH, url, REPO_DIR)
else:   # a re-run in the same session: move the existing clone to the latest commit
    _git("-C", REPO_DIR, "fetch", "-q", url, BRANCH)
    _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
_git("-C", REPO_DIR, "submodule", "update", "--init", "--recursive", "-q")
REPO_COMMIT = _git("-C", REPO_DIR, "rev-parse", "HEAD")

# No datasets pin: Kaggle now runs Python 3.13, where datasets 2.21's dependencies break.
# Audio is decoded by iqra.data (soundfile on the raw bytes), so any datasets version works.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("pip install failed:\n" + r.stdout[-1500:] + "\n" + r.stderr[-1500:])
sys.path.insert(0, f"{REPO_DIR}/src")
import warnings
warnings.filterwarnings("ignore")
import iqra.align
import datasets, numpy, pandas
print("python", sys.version.split()[0], "| datasets", datasets.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("repo commit:", REPO_COMMIT)
print("official scorer:", iqra.align.mdd_eval_dir())


In [ ]:
# ---- Inputs ----------------------------------------------------------------------------------
import json
import numpy as np
import pandas as pd
import lightgbm
from iqra import data, posteriors, scorer, verifier
from iqra.candidates import alignment, official_label_check, revert
from iqra.config import Config, find_input

cfg = Config()
SPLITS_DIR = find_input("splits.json", what="the NB11 output")
POST_DIR = find_input("post_stage1_extra_dev.pt", what="the NB14 output")
T3_DIR = find_input("candidates_stage1_extra_dev.parquet", what="the NB15 output")
OUT = "/kaggle/working/nb15_diag"
os.makedirs(OUT, exist_ok=True)
splits = data.load_splits(f"{SPLITS_DIR}/splits.json")
truth_dev = pd.read_csv(f"{SPLITS_DIR}/truth_extra_dev.csv", keep_default_na=False)
tables = {s: pd.read_parquet(f"{T3_DIR}/candidates_stage1_{s}.parquet")
          for s in ("extra_dev", "fold_a", "fold_b", "iqra_dev_slice")}
_, dev_cache = posteriors.read_cache(f"{POST_DIR}/post_stage1_extra_dev.pt")
heard = {cid: e["tokens"] for cid, e in dev_cache.items()}
expected = dict(zip(truth_dev["ID"], truth_dev["Reference_phn"].str.split()))
said = dict(zip(truth_dev["ID"], truth_dev["Annotation_phn"].str.split()))
clusters = dict(zip(truth_dev["ID"], truth_dev["Reference_phn"]))
report = {"repo_commit": REPO_COMMIT, "lightgbm": lightgbm.__version__}
print({s: len(t) for s, t in tables.items()})


In [ ]:
# ---- 1. Sentence overlap: Extra-dev vs training folds -------------------------------------------
dev_sent = set(tables["extra_dev"]["sentence"])
overlap = {s: len(dev_sent & set(tables[s]["sentence"])) for s in ("fold_a", "fold_b", "iqra_dev_slice")}
print(f"Extra-dev: {len(dev_sent)} distinct sentences | shared with fold_a / fold_b / iqra_dev_slice: {overlap}")
report["sentence_overlap"] = {"extra_dev_sentences": len(dev_sent), **overlap}


In [ ]:
# ---- 2. Label audit against the official counting ------------------------------------------------
agree = disagree = ambiguous = 0
bad = []
for cid in truth_dev["ID"]:
    for c, y, implied in official_label_check(expected[cid], said[cid], heard[cid]):
        if implied is None:
            ambiguous += 1
        elif implied == y:
            agree += 1
        else:
            disagree += 1
            bad.append((cid, c.op, c.j, c.expected, c.heard, y, implied))
print(f"label rule vs official counting: agree {agree} | disagree {disagree} | ambiguous (alignment shift) {ambiguous}")
for b in bad[:10]:
    print("  disagreement:", b)
report["label_audit"] = {"agree": agree, "disagree": disagree, "ambiguous": ambiguous}

def show(cid, c_pos, op, j):
    rows = alignment(expected[cid], heard[cid])
    exp = " ".join(f"[{t}]" if k == j else t for k, t in enumerate(expected[cid]))
    return f"{cid} {op}@{j}\n    expected: {exp}\n    said    : {' '.join(said[cid])}\n    heard   : {' '.join(heard[cid])}"

dev = tables["extra_dev"]
real_rows, false_rows = dev[dev["label"] == 1], dev[dev["label"] == 0]
sample = pd.concat([real_rows.sample(min(10, len(real_rows)), random_state=13),
                    false_rows.sample(min(10, len(false_rows)), random_state=13)])
print("\n20 labelled examples (10 real, 10 false alarms):")
for _, r in sample.iterrows():
    print(f"label={int(r['label'])}  pair={r['pair']}  type={r['err_type']}  " + show(r["clip"], r["pos"], r["op"], r["j"]))


In [ ]:
# ---- 3. Feature-group ablation (same data, same procedure as NB15) ------------------------------
train_df = pd.concat([tables["fold_a"], tables["fold_b"], tables["iqra_dev_slice"]], ignore_index=True)
assert not set(train_df["clip"]) & set(dev["clip"])
is_extra = train_df["split"].isin(["fold_a", "fold_b"]).to_numpy()
y = train_df["label"].astype(int).to_numpy()
pred_b0 = pd.DataFrame({"ID": list(heard), "Prediction": [" ".join(v) for v in heard.values()]})
c_b0 = scorer.per_clip_counts(truth_dev, pred_b0)
f1_b0 = scorer.summed_metrics(c_b0)["f1"]
taus = list(cfg.verifier.tau_grid)
ablation, probs_by = [], {}
for name, feats in verifier.FEATURE_SETS.items():
    cv = verifier.cross_validate(train_df, list(train_df["sentence"]), cfg.verifier, features=feats)
    n_rounds = max(1, int(np.median(cv["best_iterations"])))
    booster, enc = verifier.train(train_df, n_rounds, cfg.verifier, features=feats)
    probs = verifier.predict(booster, enc, dev)
    probs_by[name] = probs
    sweep = verifier.sweep_tau(dev, probs, truth_dev, heard, taus)
    tau = verifier.robust_tau(sweep, cfg.verifier.robust_tau_band)
    new = verifier.apply(dev, probs, expected, heard, tau)
    pred = pd.DataFrame({"ID": list(new), "Prediction": [" ".join(v) for v in new.values()]})
    c = scorer.per_clip_counts(truth_dev, pred)
    g = scorer.paired_bootstrap_gain(c_b0, c, clusters)
    band = sweep[(sweep["tau"] >= 0.10) & (sweep["tau"] <= 0.45)]["f1"]
    m = scorer.summed_metrics(c)
    ablation.append({"features": name, "n_features": len(feats), "cv_auc_extra": round(verifier.auc(y[is_extra], cv["oof"][is_extra]), 4),
                     "tau": tau, "P": round(m["precision"], 4), "R": round(m["recall"], 4), "F1": round(m["f1"], 4),
                     "gain_vs_B0": round(g["gain"], 4), "ci90": [round(g["lo"], 4), round(g["hi"], 4)],
                     "F1_min_tau_0.10_0.45": round(float(band.min()), 4), "F1_max_any_tau": round(float(sweep["f1"].max()), 4)})
    print(ablation[-1])
abl = pd.DataFrame(ablation)
print(f"\nB0 F1 {f1_b0:.4f}")
print(abl.to_string(index=False))
full_gain = abl.loc[abl["features"] == "full", "gain_vs_B0"].iloc[0]
noid_gain = abl.loc[abl["features"] == "no_identity", "gain_vs_B0"].iloc[0]
print(f"share of the full verifier's gain kept without identity features: {noid_gain / full_gain:.0%}")
report["ablation"] = ablation
report["B0_f1"] = round(f1_b0, 4)
report["identity_free_share_of_gain"] = round(noid_gain / full_gain, 3)


In [ ]:
# ---- 4. Deletion rule ------------------------------------------------------------------------------
def score_keep(keep_fn, label):
    keep = {}
    for r in dev[["clip", "pos", "op"]].itertuples(index=False):
        keep.setdefault(r.clip, {})[int(r.pos)] = keep_fn(r)
    new = {cid: revert(expected[cid], h, keep.get(cid, {})) for cid, h in heard.items()}
    pred = pd.DataFrame({"ID": list(new), "Prediction": [" ".join(v) for v in new.values()]})
    c = scorer.per_clip_counts(truth_dev, pred)
    g = scorer.paired_bootstrap_gain(c_b0, c, clusters)
    m = scorer.summed_metrics(c)
    row = {"system": label, "P": round(m["precision"], 4), "R": round(m["recall"], 4), "F1": round(m["f1"], 4),
           "gain_vs_B0": round(g["gain"], 4), "ci90": [round(g["lo"], 4), round(g["hi"], 4)]}
    print(row)
    return row

tau_full = abl.loc[abl["features"] == "full", "tau"].iloc[0]
p_full = dict(zip(zip(dev["clip"], dev["pos"]), probs_by["full"]))
rules = [
    score_keep(lambda r: r.op == "S", "rule: revert every deletion, keep every substitution"),
    score_keep(lambda r: r.op == "S" and p_full[(r.clip, r.pos)] >= tau_full,
               "full verifier on substitutions, every deletion reverted"),
    score_keep(lambda r: p_full[(r.clip, r.pos)] >= tau_full, "full verifier (as NB15)"),
]
report["deletion_rule"] = rules
with open(f"{OUT}/nb15_diag_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=1, default=str)


In [ ]:
# ---- PASTE BACK ----------------------------------------------------------------------------------
print(json.dumps(report, ensure_ascii=False, indent=1, default=str))
